[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/07_Differentiation.ipynb)

# Numerical Differentiation

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 5

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

## Learning goals

By the end of this notebook you should be able to:

- derive the forward, backward, and central difference formulas from a Taylor series, and state their error orders, $O(h)$ and $O(h^2)$;
- compute a second derivative with the central difference formula;
- make and interpret the log-log plot of error versus step size $h$, and explain why there is an optimal $h$ (truncation error vs round-off error);
- differentiate arrays of data with `np.diff` and `np.gradient`;
- explain why differentiating noisy data amplifies the noise, and smooth data with a moving average;
- use numerical derivatives in physics: acceleration from measured velocities, force from a potential, and the kinetic energy of a quantum wavefunction.

## 0. Setup

As usual, we import NumPy and Matplotlib first. We will also use `scipy.constants` later on.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as const

## 1. What is a derivative, numerically?

You know the definition from calculus:

$$
f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}.
$$

A computer cannot take a limit. What it *can* do is pick a small but finite $h$ and evaluate the fraction. That gives the **forward difference** approximation

$$
f'(x) \approx \frac{f(x+h) - f(x)}{h}.
$$

Geometrically, this is the slope of the line through the two points $(x, f(x))$ and $(x+h, f(x+h))$: a *secant* line that approaches the tangent line as $h \to 0$.

Let's try it on a function whose derivative we know: $f(x) = \sin x$, with $f'(x) = \cos x$.

In [ ]:
x0 = 1.0
h = 0.1

approx = (np.sin(x0 + h) - np.sin(x0)) / h
exact = np.cos(x0)

print(f"forward difference: {approx:.8f}")
print(f"exact cos(1):       {exact:.8f}")
print(f"error:              {abs(approx - exact):.2e}")

Not bad, but not great: the error is about 0.04. Let's make $h$ smaller and watch the error.

In [ ]:
for h in [0.1, 0.01, 0.001, 0.0001]:
    approx = (np.sin(x0 + h) - np.sin(x0)) / h
    print(f"h = {h:<8}  error = {abs(approx - exact):.2e}")

Each time $h$ shrinks by a factor of 10, the error shrinks by about a factor of 10. The error is *proportional to $h$*. We write this as "the error is $O(h)$" (read: "order $h$"). The next section explains where that comes from.

## 2. Finite differences from Taylor series

### 2.1 Forward difference

The Taylor series of $f$ around $x$ is

$$
f(x+h) = f(x) + h f'(x) + \frac{h^2}{2} f''(x) + \frac{h^3}{6} f^{(3)}(x) + \cdots
$$

Solve for $f'(x)$:

$$
f'(x) = \frac{f(x+h) - f(x)}{h} - \frac{h}{2} f''(x) - \cdots
$$

So the forward difference is exact except for an error term $-\tfrac{h}{2}f''(x)$ plus smaller terms. This error, which comes from chopping off the Taylor series, is called the **truncation error**. It is $O(h)$.

### 2.2 Backward difference

Expanding in the other direction,

$$
f(x-h) = f(x) - h f'(x) + \frac{h^2}{2} f''(x) - \frac{h^3}{6} f^{(3)}(x) + \cdots
$$

gives the **backward difference**

$$
f'(x) \approx \frac{f(x) - f(x-h)}{h}, \qquad \text{error} = +\frac{h}{2}f''(x) + \cdots = O(h).
$$

### 2.3 Central difference

Now subtract the two expansions. The $f(x)$ terms cancel, and so do the $h^2$ terms:

$$
f(x+h) - f(x-h) = 2h f'(x) + \frac{h^3}{3} f^{(3)}(x) + \cdots
$$

Dividing by $2h$:

$$
\boxed{f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}}, \qquad \text{error} = -\frac{h^2}{6} f^{(3)}(x) + \cdots = O(h^2).
$$

This is the **central difference**. It costs the same (two function evaluations) as the forward difference, but its error is $O(h^2)$: halve $h$ and the error drops by a factor of 4; shrink $h$ by 10 and the error drops by 100. The forward and backward errors have opposite signs, and the central difference is their average, so the leading errors cancel.

### 2.4 Writing the formulas as functions

Here is something new: in Python you can pass a **function as an argument** to another function, just like a number or an array. Inside `forward_diff`, the name `f` refers to whatever function you passed in (`np.sin`, or one you wrote yourself), and `f(x + h)` calls it.

In [ ]:
def forward_diff(f, x, h):
    """Forward difference estimate of f'(x). Error O(h)."""
    return (f(x + h) - f(x)) / h

def backward_diff(f, x, h):
    """Backward difference estimate of f'(x). Error O(h)."""
    return (f(x) - f(x - h)) / h

def central_diff(f, x, h):
    """Central difference estimate of f'(x). Error O(h^2)."""
    return (f(x + h) - f(x - h)) / (2 * h)

In [ ]:
print(f"{'h':>8} {'forward':>10} {'backward':>10} {'central':>10}")
for h in [0.1, 0.01, 0.001]:
    ef = abs(forward_diff(np.sin, x0, h) - exact)
    eb = abs(backward_diff(np.sin, x0, h) - exact)
    ec = abs(central_diff(np.sin, x0, h) - exact)
    print(f"{h:>8} {ef:>10.2e} {eb:>10.2e} {ec:>10.2e}")

The forward and backward errors fall by 10 per row; the central error falls by 100 per row. With $h = 0.001$, the central difference is already accurate to about 7 significant figures.

You can also check the prediction for the *size* of the error. For the forward difference we predicted error $\approx \tfrac{h}{2}|f''(x)| = \tfrac{h}{2}\sin(1)$. With $h = 0.01$ that is $0.0042$. Compare with the table above.

In [ ]:
h = 0.01
print("predicted forward error:", h / 2 * np.sin(x0))
print("predicted central error:", h**2 / 6 * np.cos(x0))

### Your turn 1

Use `forward_diff` and `central_diff` to estimate the derivative of $f(x) = e^{x}$ at $x = 2$ (exact answer $e^2$) for $h = 10^{-1}, 10^{-2}, 10^{-3}$. Print the errors and check that they scale as $O(h)$ and $O(h^2)$. Then write your own function `g(x)` for $g(x) = x^3$ and repeat at $x = 1$. Is the central difference for $x^3$ exactly right? Use the error formula $-\frac{h^2}{6}f^{(3)}(x)$ to explain what you see.

In [ ]:
# Your code here

## 3. The key plot: error vs step size

From the Taylor series, smaller $h$ always means smaller error. But you learned in the last notebook that computers store numbers with only about 16 significant digits (machine epsilon $\varepsilon \approx 2.2\times 10^{-16}$). Look at the numerator $f(x+h) - f(x)$: when $h$ is tiny, these two numbers are almost equal, and subtracting them is **catastrophic cancellation**.

Let's scan $h$ over 16 orders of magnitude and see what really happens. `np.logspace(-16, 0, 161)` makes 161 values evenly spaced on a log scale, from $10^{-16}$ to $10^0$.

In [ ]:
hs = np.logspace(-16, 0, 161)

err_fwd = np.abs(forward_diff(np.sin, x0, hs) - exact)
err_cen = np.abs(central_diff(np.sin, x0, hs) - exact)

print(hs[:3])
print(err_fwd[:3])

Notice that we passed the whole *array* `hs` into `forward_diff`. Because the function uses only arithmetic and `np.sin`, it works element by element on arrays automatically.

Now plot both errors on log-log axes. On a log-log plot a power law $\text{error} \propto h^n$ is a straight line with slope $n$, so we add dashed reference lines with slopes 1 and 2.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(hs, err_fwd, ".", label="forward difference")
ax.loglog(hs, err_cen, ".", label="central difference")
ax.loglog(hs, 0.5 * hs, "k--", lw=1, label=r"slope 1 ($\propto h$)")
ax.loglog(hs, 0.1 * hs**2, "k:", lw=1, label=r"slope 2 ($\propto h^2$)")
ax.set_xlabel("step size $h$")
ax.set_ylabel(r"absolute error in $f'(1)$")
ax.set_ylim(1e-12, 1e1)
ax.legend()
ax.set_title(r"Derivative of $\sin x$ at $x=1$")
plt.show()

This is the most important figure of the week. Read it from right to left (decreasing $h$):

1. **Large $h$ (right side):** the error follows the reference lines: slope 1 for forward, slope 2 for central. This is truncation error, exactly as the Taylor series predicted.
2. **Small $h$ (left side):** the error *grows* as $h$ shrinks, and it is noisy. This is **round-off error**.
3. In between, there is a minimum: the **optimal step size**.

Making $h$ smaller is *not* always better.

### 3.1 Estimating the optimal $h$

Each value $f(x)$ is stored with a relative error of about $\varepsilon$, so the numerator $f(x+h) - f(x)$ carries an absolute error of roughly $\varepsilon |f(x)|$. Dividing by $h$ gives a round-off error of about $\varepsilon|f|/h$, which grows as $h$ shrinks. For the forward difference, the total error is about

$$
E(h) \approx \underbrace{\frac{h}{2}|f''|}_{\text{truncation}} + \underbrace{\frac{2\varepsilon |f|}{h}}_{\text{round-off}}.
$$

Setting $dE/dh = 0$ gives $h_\text{opt} = 2\sqrt{\varepsilon|f|/|f''|}$. If $f$ and its derivatives are all of order 1, then

$$
h_\text{opt} \sim \sqrt{\varepsilon} \approx 10^{-8} \quad \text{(forward)}, \qquad E_\text{min} \sim \sqrt{\varepsilon} \approx 10^{-8}.
$$

For the central difference the truncation term is $\tfrac{h^2}{6}|f^{(3)}|$ instead, and the same calculation gives

$$
h_\text{opt} \sim \varepsilon^{1/3} \approx 10^{-5} \quad \text{(central)}, \qquad E_\text{min} \sim \varepsilon^{2/3} \approx 10^{-11}.
$$

Let's check these against the plot.

In [ ]:
eps = np.finfo(float).eps
print(f"machine epsilon      = {eps:.2e}")
print(f"sqrt(eps)            = {eps**0.5:.1e}")
print(f"eps^(1/3)            = {eps**(1/3):.1e}")

i_f = np.argmin(err_fwd)
i_c = np.argmin(err_cen)
print(f"forward: best h = {hs[i_f]:.1e}, smallest error = {err_fwd[i_f]:.1e}")
print(f"central: best h = {hs[i_c]:.1e}, smallest error = {err_cen[i_c]:.1e}")

`np.argmin(arr)` returns the *index* of the smallest element, which we then use to look up the matching `h`.

The estimates agree with the experiment to within about an order of magnitude, which is all you should expect from this kind of rough reasoning (we ignored factors of order 1, and round-off errors are partly random). The practical lessons:

- With the central difference and $h \approx 10^{-5}$ (times the typical scale of $x$), you can get about 10 or more correct digits, but not 16.
- **Never** use $h = 10^{-15}$ thinking "smaller is better."
- If $x$ is not of order 1 (say $x$ is a distance of $10^{-10}$ m), scale $h$ to match: use $h \approx 10^{-5}\,|x|$ or $10^{-5}$ times the length scale over which $f$ changes.

### Your turn 2

Make the error-vs-$h$ plot for the **backward** difference and for a different function, $f(x) = e^x$ at $x = 1$. Find the optimal $h$ for the backward and central differences with `np.argmin` and compare them to $\sqrt{\varepsilon}$ and $\varepsilon^{1/3}$. Remember to label your axes and include a legend.

In [ ]:
# Your code here

## 4. Second derivatives

Many physics equations involve second derivatives: acceleration $\ddot x$, the kinetic-energy term $-\frac{\hbar^2}{2m}\psi''$ in the Schrödinger equation, the wave equation. *Add* the two Taylor expansions from Section 2 instead of subtracting them. Now the odd terms cancel:

$$
f(x+h) + f(x-h) = 2f(x) + h^2 f''(x) + \frac{h^4}{12} f^{(4)}(x) + \cdots
$$

so

$$
\boxed{f''(x) \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2}}, \qquad \text{error} = O(h^2).
$$

In [ ]:
def second_diff(f, x, h):
    """Central difference estimate of f''(x). Error O(h^2)."""
    return (f(x + h) - 2 * f(x) + f(x - h)) / h**2

exact2 = -np.sin(x0)          # (sin x)'' = -sin x
for h in [0.1, 0.01, 0.001]:
    err = abs(second_diff(np.sin, x0, h) - exact2)
    print(f"h = {h:<6}  error = {err:.2e}")

The error drops by 100 per factor-of-10 step, as promised. But the round-off problem is now *worse*: we divide by $h^2$ rather than $h$, so the round-off error is about $\varepsilon|f|/h^2$. Balancing that against $h^2|f^{(4)}|/12$ gives $h_\text{opt} \sim \varepsilon^{1/4} \approx 10^{-4}$, with only about 8 correct digits at best.

In [ ]:
err_2nd = np.abs(second_diff(np.sin, x0, hs) - exact2)

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(hs, err_2nd, ".", color="C2", label="second derivative (central)")
ax.loglog(hs, 0.05 * hs**2, "k:", lw=1, label=r"slope 2 ($\propto h^2$)")
ax.axvline(eps**0.25, color="gray", lw=1, label=r"$\varepsilon^{1/4}$")
ax.set_xlabel("step size $h$")
ax.set_ylabel(r"absolute error in $f''(1)$")
ax.set_ylim(1e-12, 1e4)
ax.legend()
plt.show()

## 5. Richardson extrapolation (brief)

Here is a neat trick. We know the central difference $D(h)$ has the form

$$
D(h) = f'(x) + c\,h^2 + O(h^4)
$$

with some unknown constant $c$. Compute it twice, with $h$ and $h/2$:

$$
D(h/2) = f'(x) + c\,\frac{h^2}{4} + O(h^4).
$$

Multiply the second equation by 4 and subtract the first: the $c\,h^2$ terms cancel, leaving

$$
f'(x) \approx \frac{4D(h/2) - D(h)}{3}, \qquad \text{error } O(h^4).
$$

This is **Richardson extrapolation**: combine two low-accuracy answers whose error you know the *form* of, and cancel the leading error. The same idea appears again next week for integration.

In [ ]:
def richardson_diff(f, x, h):
    """Richardson-extrapolated central difference. Error O(h^4)."""
    return (4 * central_diff(f, x, h / 2) - central_diff(f, x, h)) / 3

for h in [0.1, 0.05, 0.025]:
    ec = abs(central_diff(np.sin, x0, h) - exact)
    er = abs(richardson_diff(np.sin, x0, h) - exact)
    print(f"h = {h:<6} central error = {ec:.2e}   Richardson error = {er:.2e}")

Halving $h$ cuts the Richardson error by about $2^4 = 16$. With $h = 0.1$ it already beats the plain central difference with $h = 0.001$.

## 6. Derivatives of data: `np.diff` and `np.gradient`

In the lab, you usually do not have a formula $f(x)$. You have *measurements* at discrete points: positions $x_0, x_1, \ldots$ at times $t_0, t_1, \ldots$. You cannot choose $h$; it is the spacing of your data.

Let's simulate a simple measurement: a mass on a spring. Quick physics recap: Hooke's law says the spring pushes back with force $F = -kx$, and Newton's second law $F = ma$ then gives $a = -(k/m)\,x$. The solution is **simple harmonic motion**, $x(t) = A\cos(\omega t)$ with angular frequency $\omega = \sqrt{k/m}$. Its velocity is $v = dx/dt = -A\omega\sin(\omega t)$ and its acceleration is $a = dv/dt = -\omega^2 x$. We will pretend we measured $x$ every 0.1 s and try to recover $v$ and $a$ from the numbers alone.

In [ ]:
A = 0.05            # amplitude (m)
omega = 2.0         # angular frequency (rad/s)
t = np.linspace(0, 10, 101)     # times (s), spacing 0.1 s
x = A * np.cos(omega * t)       # positions (m)

dt = t[1] - t[0]
print("time step dt =", dt, "s")

### 6.1 `np.diff`

`np.diff(a)` returns the differences between neighboring elements, `a[1] - a[0], a[2] - a[1], ...`. Note that the result is **one element shorter** than the input.

In [ ]:
print(np.diff(np.array([1, 4, 9, 16, 25])))

v_diff = np.diff(x) / np.diff(t)
print(len(x), len(v_diff))

`np.diff(x) / np.diff(t)` is the forward (or backward) difference at every point. Each value is the slope over one interval, so it is most accurate at the *midpoint* of that interval. In fact, if you assign it to the midpoint time, it is a central difference with step $\Delta t/2$, accurate to $O(\Delta t^2)$.

### 6.2 `np.gradient`

`np.gradient(x, t)` does the bookkeeping for you. It uses the central difference $\frac{x_{i+1} - x_{i-1}}{t_{i+1} - t_{i-1}}$ at interior points and one-sided differences at the two ends, and it returns an array the **same length** as the input. The second argument can be either the spacing `dt` or the array of times `t` (which also handles uneven spacing).

In [ ]:
v_grad = np.gradient(x, t)              # velocity (m/s), same length as t
t_mid = 0.5 * (t[1:] + t[:-1])          # midpoints of each interval
v_exact = -A * omega * np.sin(omega * t)

fig, ax = plt.subplots()
ax.plot(t, v_exact, "k-", label="exact")
ax.plot(t, v_grad, "o", ms=3, label="np.gradient")
ax.plot(t_mid, v_diff, "x", ms=4, label="np.diff at midpoints")
ax.set_xlabel("time $t$ (s)")
ax.set_ylabel("velocity $v$ (m/s)")
ax.legend()
plt.show()

All three agree well on this scale. To get the acceleration, apply `np.gradient` again. Compare with the exact $a = -\omega^2 x$.

In [ ]:
a_grad = np.gradient(v_grad, t)
a_exact = -omega**2 * x

print("max |error| in v:", np.max(np.abs(v_grad - v_exact)), "m/s")
print("max |error| in a:", np.max(np.abs(a_grad - a_exact)), "m/s^2")
print("max |error| in a, interior only:", np.max(np.abs(a_grad - a_exact)[2:-2]), "m/s^2")

The errors are largest at the two endpoints, where `np.gradient` must use the less accurate one-sided formulas. This is a common thing to keep in mind: **numerical derivatives are worst at the edges of your data.** (You can ask for second-order one-sided formulas at the edges with `np.gradient(x, t, edge_order=2)`.)

## 7. Real data: velocity to acceleration

The file `velocities.txt` (from Newman, *Computational Physics*, Exercise 5.1) contains measurements of the $x$-velocity of a particle: column 0 is time in seconds, column 1 is velocity in m/s. First the standard data-loading helper:

In [ ]:
import os
DATA_URL = "https://raw.githubusercontent.com/burzynski-lab/OU_PHYS2222_ComputationalPhysics/main/Data/"

def data_path(filename):
    """Return a local path to a course data file if it exists, otherwise its GitHub URL."""
    for folder in ["../Data", "Data", "."]:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return DATA_URL + filename

In [ ]:
data = np.loadtxt(data_path("velocities.txt"))
t_data = data[:, 0]     # time (s)
v_data = data[:, 1]     # velocity (m/s)
print(data.shape)
print(data[:5])

In [ ]:
a_data = np.gradient(v_data, t_data)    # acceleration (m/s^2)

fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
axes[0].plot(t_data, v_data, ".-")
axes[0].set_ylabel("velocity (m/s)")
axes[1].plot(t_data, a_data, ".-", color="C1")
axes[1].axhline(0, color="gray", lw=0.8)
axes[1].set_ylabel(r"acceleration (m/s$^2$)")
axes[1].set_xlabel("time (s)")
plt.show()

`sharex=True` makes the two panels share one time axis. Notice that the acceleration is zero exactly where the velocity has a maximum or minimum, as it should be. The acceleration curve is also visibly more jagged than the velocity curve. That is our next topic.

## 8. Differentiating noisy data

Real measurements have noise. Suppose each position has a small random error $\delta$ of typical size $\sigma$. The central difference becomes

$$
\frac{(x_{i+1} + \delta_{i+1}) - (x_{i-1} + \delta_{i-1})}{2h} = \frac{x_{i+1} - x_{i-1}}{2h} + \underbrace{\frac{\delta_{i+1} - \delta_{i-1}}{2h}}_{\sim\, \sigma/h}.
$$

The noise in the derivative is about $\sigma/h$, so **finer sampling makes the noise in the derivative worse**. It is the same story as round-off error, except that the "$\varepsilon$" is now your measurement precision, which is much larger than $10^{-16}$. For a second derivative, the noise grows like $\sigma/h^2$.

Let's add 1 mm of simulated Gaussian noise to the spring positions and differentiate.

In [ ]:
rng = np.random.default_rng(42)
sigma = 0.001                                   # 1 mm noise (simulated)
x_noisy = x + rng.normal(0, sigma, size=len(x))

v_noisy = np.gradient(x_noisy, t)
a_noisy = np.gradient(v_noisy, t)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True)
axes[0].plot(t, x_noisy, ".", ms=3, label="noisy data")
axes[0].plot(t, x, "k-", lw=1, label="true")
axes[0].set_ylabel("$x$ (m)")
axes[1].plot(t, v_noisy, ".-", ms=3, label="from noisy data")
axes[1].plot(t, v_exact, "k-", lw=1, label="true")
axes[1].set_ylabel("$v$ (m/s)")
axes[2].plot(t, a_noisy, ".-", ms=3, label="from noisy data")
axes[2].plot(t, a_exact, "k-", lw=1, label="true")
axes[2].set_ylabel(r"$a$ (m/s$^2$)")
axes[2].set_xlabel("time $t$ (s)")
for ax in axes:
    ax.legend(loc="upper right", fontsize=8)
plt.show()

The position data look almost perfect: 1 mm of noise on a 5 cm oscillation is barely visible. The velocity is noticeably noisy, and the acceleration is much worse. Every derivative amplifies the noise.

### 8.1 A simple remedy: smoothing with a moving average

One fix is to **smooth** the data before differentiating. The simplest smoother is a **moving average**: replace each point by the average of itself and its neighbors, in a window of $w$ points. Averaging $w$ independent noise values reduces the noise by about $\sqrt{w}$.

`np.convolve(a, kernel, mode="same")` slides the `kernel` array along `a` and computes a weighted sum at each position. With `kernel = np.ones(w) / w` (all weights equal to $1/w$), that weighted sum is just the average of $w$ neighbors. `mode="same"` returns an array the same length as `a`, but the first and last few points are averaged with zeros outside the data, so they are wrong; we will ignore the edges.

In [ ]:
def moving_average(a, w):
    """Average of each point with its neighbors in a window of w points (w odd)."""
    kernel = np.ones(w) / w
    return np.convolve(a, kernel, mode="same")

w = 7
x_smooth = moving_average(x_noisy, w)
v_smooth = moving_average(np.gradient(x_smooth, t), w)
a_smooth = np.gradient(v_smooth, t)

edge = w        # number of unreliable points at each end

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, a_noisy, ".", ms=3, color="lightgray", label="raw noisy derivative")
ax.plot(t[edge:-edge], a_smooth[edge:-edge], "-", color="C3", label=f"smoothed (w = {w})")
ax.plot(t, a_exact, "k--", lw=1, label="true")
ax.set_xlabel("time $t$ (s)")
ax.set_ylabel(r"acceleration (m/s$^2$)")
ax.set_ylim(-0.5, 0.5)
ax.legend()
plt.show()

Much better, but look closely: smoothing also slightly reduces the amplitude of the true signal, because averaging over a window flattens peaks. Choosing $w$ is a trade-off between noise (small $w$) and distortion (large $w$), just like choosing $h$ earlier. There are better smoothers (for example, fitting a low-order polynomial in each window, the Savitzky–Golay filter in `scipy.signal.savgol_filter`), and fitting a model to the data, which we will cover in Unit 3, is often the best approach of all.

### Your turn 3

Repeat the noisy-data experiment above with the sampling made 5 times finer (`t = np.linspace(0, 10, 501)`) but the same noise level $\sigma = 1$ mm. Compute the root-mean-square error of the raw `np.gradient` velocity, `np.sqrt(np.mean((v_noisy - v_exact)**2))`, for both samplings. Does finer sampling help or hurt? By roughly what factor, and does that match the $\sigma/h$ estimate?

In [ ]:
# Your code here

## 9. Physics application: force from a potential

In classical mechanics, a conservative force is minus the gradient of the potential energy: $F(r) = -\dfrac{dV}{dr}$. The **Lennard-Jones potential** is a standard model for the interaction between two neutral atoms:

$$
V(r) = 4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right].
$$

The $r^{-12}$ term is a strong short-range repulsion and the $r^{-6}$ term is the van der Waals attraction. For argon, $\epsilon \approx 0.0104$ eV and $\sigma \approx 0.340$ nm. We will work in eV and nm, so the force comes out in eV/nm.

In [ ]:
eps_LJ = 0.0104     # well depth (eV), argon
sig_LJ = 0.340      # length scale (nm), argon

def V_LJ(r):
    """Lennard-Jones potential energy (eV) at separation r (nm)."""
    return 4 * eps_LJ * ((sig_LJ / r)**12 - (sig_LJ / r)**6)

def F_LJ_exact(r):
    """Exact force -dV/dr (eV/nm)."""
    return 4 * eps_LJ * (12 * sig_LJ**12 / r**13 - 6 * sig_LJ**6 / r**7)

In [ ]:
r = np.linspace(0.32, 0.8, 300)        # separations (nm)
h = 1e-5 * sig_LJ                      # step scaled to the length scale of the problem
F_num = -central_diff(V_LJ, r, h)

print("max relative error:", np.max(np.abs(F_num - F_LJ_exact(r)) / np.abs(F_LJ_exact(r)).max()))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
axes[0].plot(r, V_LJ(r) * 1000, "C0")                   # convert eV -> meV
axes[0].set_ylabel("$V(r)$ (meV)")
axes[0].set_ylim(-12, 10)
axes[1].plot(r, F_num * 1000, "C1", label="numerical $-dV/dr$")
axes[1].plot(r[::15], F_LJ_exact(r[::15]) * 1000, "k.", label="exact")
axes[1].set_ylabel("$F(r)$ (meV/nm)")
axes[1].set_ylim(-150, 150)
axes[1].set_xlabel("separation $r$ (nm)")
axes[1].legend()
for ax in axes:
    ax.axhline(0, color="gray", lw=0.8)
plt.show()

(Recap: a positive force pushes the atoms apart, a negative one pulls them together. The force is the *downhill slope* of the potential energy curve, like a ball rolling on a hill.)

The force is positive (repulsive) at short distances, negative (attractive) at long distances, and zero at the bottom of the potential well, the equilibrium separation $r_0 = 2^{1/6}\sigma \approx 0.382$ nm. We will find that zero numerically in the root-finding notebook.

Why bother with a numerical derivative when we can differentiate this by hand? In research, potentials are often given as complicated formulas or as tables from other calculations, and a numerical derivative is quick and, used carefully, accurate.

## 10. Physics application: kinetic energy of a quantum state

**Physics recap.** In quantum mechanics a particle is described by a wavefunction $\psi(x)$, and $|\psi(x)|^2$ is the probability density for finding it at $x$. The time-independent Schrödinger equation is

$$
-\frac{\hbar^2}{2m}\frac{d^2\psi}{dx^2} + V(x)\,\psi = E\,\psi .
$$

The first term is the kinetic energy acting on $\psi$: the kinetic energy operator is $\hat T = -\dfrac{\hbar^2}{2m}\dfrac{d^2}{dx^2}$. (An *operator* is just a rule that turns one function into another; here the rule is "take the second derivative and multiply by $-\hbar^2/2m$.") The left side as a whole is written $\hat H\psi$, where $\hat H = \hat T + V$ is the **Hamiltonian** (total-energy operator). The **average** (expectation) value of the kinetic energy is $\langle T\rangle = \int \psi^*\,\hat T\psi\,dx$. So computing kinetic energies needs a second derivative of the wavefunction, which is exactly what we just learned.

The quantum harmonic oscillator has potential $V(x) = \tfrac12 m\omega^2 x^2$ (a mass on a spring, as in Section 6). Take its ground state (lowest-energy state). To keep the numbers simple we use **natural units** where $\hbar = m = \omega = 1$, so lengths are measured in units of $\sqrt{\hbar/m\omega}$ and energies in units of $\hbar\omega$. Then

$$
\psi_0(x) = \pi^{-1/4} e^{-x^2/2}, \qquad \hat H \psi_0 = -\frac{1}{2}\psi_0'' + \frac{1}{2}x^2\psi_0 = E_0 \psi_0, \qquad E_0 = \tfrac{1}{2}.
$$

### 10.1 Checking the Schrödinger equation numerically

If $\psi_0$ really is an eigenstate, then $\hat H\psi_0 / \psi_0$ must equal the constant $E_0 = 1/2$ at *every* $x$. Let's check with a grid of points and `np.gradient` applied twice.

In [ ]:
x_grid = np.linspace(-5, 5, 1001)       # in units of sqrt(hbar/m omega)
dx = x_grid[1] - x_grid[0]
psi = np.pi**(-0.25) * np.exp(-x_grid**2 / 2)

d2psi = np.gradient(np.gradient(psi, dx), dx)
H_psi = -0.5 * d2psi + 0.5 * x_grid**2 * psi
local_E = H_psi / psi

print("H psi / psi at a few points:", local_E[[100, 300, 500, 700, 900]])

It is very close to $0.5$ everywhere; the small deviations far out in the tails ($|x| \approx 4$) come from dividing by a tiny $\psi$ there. (A wavefunction that is *not* an eigenstate would give an $x$-dependent ratio. Try it in the exercise below.)

### 10.2 Expectation value of the kinetic energy

The expectation value $\langle T \rangle = \int \psi^*\,\hat T\,\psi\,dx$ requires an integral. Next week we will learn proper integration methods; for now, the simplest approximation is a sum, $\int g(x)\,dx \approx \sum_i g(x_i)\,\Delta x$, which works very well here because $\psi$ goes smoothly to zero at the ends of the grid. The virial theorem predicts $\langle T\rangle = \tfrac{1}{2}E_0 = \tfrac{1}{4}\hbar\omega$.

In [ ]:
T_expect = np.sum(psi * (-0.5 * d2psi)) * dx
print(f"<T> = {T_expect:.6f} hbar*omega   (expected 0.25)")

Now convert to real units for an electron in a harmonic trap with $\hbar\omega = 1$ eV. Since $\langle T\rangle$ is in units of $\hbar\omega$, we just multiply. We can also find the size of the natural length unit $\sqrt{\hbar/m\omega}$, which tells you roughly how spread out the electron is. (`const.e` is the elementary charge, used to convert eV to joules.)

In [ ]:
hbar_omega_eV = 1.0
print(f"<T> = {T_expect * hbar_omega_eV:.4f} eV")
omega_e = hbar_omega_eV * const.e / const.hbar          # angular frequency (rad/s)
length = np.sqrt(const.hbar / (const.m_e * omega_e))    # length unit sqrt(hbar/(m omega)), in m
print(f"length unit = {length * 1e9:.3f} nm")

### Your turn 4

The function $\phi(x) = (2/\pi)^{1/4} e^{-x^2}$ is normalized but is **not** the ground state of the oscillator (it is too narrow). In the same natural units:

1. Compute and plot $\hat H\phi/\phi$ versus $x$ on $-3 \le x \le 3$. Is it constant?
2. Compute $\langle T\rangle$ and $\langle H\rangle = \int \phi\,\hat H \phi\,dx$ using sums. Check that $\langle H\rangle > E_0 = 1/2$, as the variational principle requires.

In [ ]:
# Your code here

### Your turn 5

A free-particle Gaussian wave packet moving to the right is $\psi(x) = (2\pi s^2)^{-1/4} e^{-x^2/(4s^2)} e^{ik_0 x}$. The momentum operator is $\hat p = -i\hbar\, d/dx$. Using $\hbar = 1$, $s = 1$, $k_0 = 5$:

1. Build `psi` on a grid from $-10$ to $10$. (In Python the imaginary unit is written `1j`, so $e^{ik_0x}$ is `np.exp(1j * k0 * x)`. NumPy handles complex arrays automatically.)
2. Compute $\langle p\rangle = \int \psi^*\,(-i\,\psi')\,dx$ with `np.gradient` and a sum. Use `np.conj(psi)` for $\psi^*$, and print the real part with `.real`. Do you get $\hbar k_0$?
3. How does your answer depend on the grid spacing? Try 201 and 2001 points.

In [ ]:
# Your code here

## Summary

- **Forward difference:** $f'(x) \approx \dfrac{f(x+h) - f(x)}{h}$, error $O(h)$. **Backward** is the mirror image, also $O(h)$.
- **Central difference:** $f'(x) \approx \dfrac{f(x+h) - f(x-h)}{2h}$, error $O(h^2)$. Use it by default.
- **Second derivative:** $f''(x) \approx \dfrac{f(x+h) - 2f(x) + f(x-h)}{h^2}$, error $O(h^2)$.
- All are derived from Taylor series; the neglected terms are the **truncation error**.
- **Round-off error** grows like $\varepsilon/h$ (or $\varepsilon/h^2$ for $f''$). The total error has a minimum at $h_\text{opt} \sim \sqrt\varepsilon \approx 10^{-8}$ (forward) or $\varepsilon^{1/3} \approx 10^{-5}$ (central), scaled to the problem. Smaller $h$ is not always better.
- **Richardson extrapolation** combines $D(h)$ and $D(h/2)$ to cancel the leading error term.
- For data: `np.diff(a)` gives neighbor differences (one shorter); `np.gradient(y, x)` gives central differences, same length, one-sided at the edges.
- Differentiation **amplifies noise** by about $1/h$ per derivative. Smooth first (e.g. a moving average with `np.convolve`) or fit a model.
- New Python: passing functions as arguments, `np.logspace`, `np.argmin`, `np.diff`, `np.gradient`, `np.convolve`, complex numbers with `1j`.

## Further reading

- M. Newman, *Computational Physics*, Chapter 5.10 (derivatives), free chapters at <https://websites.umich.edu/~mejn/cp/>.
- NumPy documentation: [`numpy.gradient`](https://numpy.org/doc/stable/reference/generated/numpy.gradient.html), [`numpy.diff`](https://numpy.org/doc/stable/reference/generated/numpy.diff.html).
- SciPy documentation: [`scipy.signal.savgol_filter`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.savgol_filter.html) for smoothing noisy data.